# 01 · Dataset (Colab)

The harmonised, leakage-safe dataset (fetch → label harmonisation → normalise → near-duplicate grouping → splits).

**Option A (normal):** the dataset was built on a PC and packed into zips on Drive (`govi-ai/dataset/`).
Run the first three cells: they unpack it onto Colab's fast local disk.

**Option B:** build it on Colab from scratch (CPU runtime is enough, ~1–2 h).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GOVI = '/content/drive/MyDrive/govi-ai'
DATA_ROOT = '/content/data'   # local disk: fast reads during training

In [ ]:
!git clone -q https://github.com/Kalana-Lakshan/govi-isuru-ai-service.git /content/govi-isuru-ai-service || git -C /content/govi-isuru-ai-service pull -q
%cd /content/govi-isuru-ai-service
!pip install -q -e ".[data]"

## Option A · unpack the prebuilt dataset from Drive

In [ ]:
!python scripts/pack_dataset.py --unpack --out $GOVI/dataset --data-root $DATA_ROOT
import json, pandas as pd
report = json.load(open(f'{DATA_ROOT}/build_report.json'))
print(json.dumps({k: report[k] for k in ['rows', 'by_role', 'by_split', 'dedupe', 'known_classes_without_images']}, indent=1))
m = pd.read_csv(f'{DATA_ROOT}/manifest.csv')
m[m.role == 'known'].pivot_table(index=['crop', 'label'], columns='source', values='sample_id', aggfunc='count', fill_value=0)

## Option B · build on Colab
Manual sources: put the CS-D archive contents in `$GOVI/raw/csd_assam/`; legacy v1 folders in `$DATA_ROOT/legacy/`.
Re-runnable: sources that already have `manifests/<id>.csv` are skipped (use `--force` to rebuild one).

In [ ]:
!python scripts/build_dataset.py --data-root $DATA_ROOT --raw-root $GOVI/raw
!python scripts/pack_dataset.py --data-root $DATA_ROOT --out $GOVI/dataset

## Legacy baseline (the paper's "before" numbers)
Needs the v1 `ai-service/models/` folder at `$GOVI/legacy_models/`.

In [ ]:
!pip install -q -e ".[legacy]"
!python scripts/eval_legacy.py --data-root $DATA_ROOT --legacy-models $GOVI/legacy_models --out $GOVI/artifacts/legacy_baseline.json